# Carnet 1 : Benchmark Avancé des Modèles

## Contexte du Projet
On cherche à prédire la **gravité des accidents de la route** (`grav_binary`) selon la météo, le type de route, les véhicules impliqués, etc. L'idée de ce premier carnet est de lancer plusieurs modèles en même temps (un "benchmark") pour voir lequel s'en sort le mieux par défaut, avant de l'optimiser.

Comme on s'attaque à un problème de classification (prédire si un accident est grave ou non), on ne va pas choisir un algorithme au hasard. L'idée d'un benchmark, c'est justement de tester plusieurs approches très différentes (de la simple régression logistique jusqu'aux modèles plus musclés de type « ensemble ») pour voir ce qui accroche le mieux à nos données. Ça nous permet de justifier nos choix pour la suite plutôt que d'y aller à l'aveuglette !

---

## Sommaire
* [Préparation](#Préparation)
* [Entraînement et Suivi (Tracking)](#Entraînement-et-Suivi-\(Tracking\))
* [Conclusion](#Conclusion)


## Préparation
### Imports et Configuration MLflow


In [ ]:
import pandas as pd
import mlflow
import mlflow.sklearn
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# On connecte le notebook à notre base MLflow locale
TRACKING_URI = "sqlite:///mlflow.db"
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment("1_Benchmark_Modeles_Complet")


### Chargement des données


In [ ]:
df = pd.read_csv('data/dataset_accident.csv', sep=';')
X = df.drop(columns=["grav_binary", "grav_ordered"])
y = df["grav_binary"]

print(f"Dimensions du dataset : {X.shape}")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## Entraînement et Suivi (Tracking)
### Lancement du Benchmark multi-modèles
On va entraîner 4 modèles classiques. MLflow va automatiquement se souvenir de leurs paramètres de base (ce qui nous évitera de les noter) et calculer 4 scores de performance pour qu'on puisse les comparer proprement.


In [ ]:
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(n_estimators=50, random_state=42),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=50, random_state=42),
    "XGBoost": xgb.XGBClassifier(n_estimators=50, max_depth=5, use_label_encoder=False, eval_metric='logloss', random_state=42)
}

print("-- Lancement du Benchmark --")

for desc_name, model in models.items():
    # On crée une "boîte" (run) par modèle dans MLflow
    with mlflow.start_run(run_name=f"Base_{desc_name}") as run:
        print(f"\n-- Entraînement de {desc_name} --")
        
        # On enregistre la configuration du modèle
        mlflow.log_param("model_type", desc_name)
        mlflow.log_params(model.get_params())
        
        # 1. On entraîne
        model.fit(X_train, y_train)
        
        # 2. On fait les prédictions
        preds = model.predict(X_test)
        
        # 3. On calcule tous les scores
        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds, average='weighted')
        prec = precision_score(y_test, preds, average='weighted', zero_division=0)
        rec = recall_score(y_test, preds, average='weighted', zero_division=0)
        
        # 4. On envoie les notes à MLflow
        mlflow.log_metrics({
            "accuracy": acc,
            "f1_score": f1,
            "precision": prec,
            "recall": rec
        })
        print(f"  Score Accuracy : {acc:.4f} | F1 : {f1:.4f}")
        
        # 5. On place le modèle entraîné dans les artefacts de MLflow
        mlflow.sklearn.log_model(model, artifact_path=desc_name)

print("\n-- Benchmark terminé. Résultats disponibles sur le serveur MLflow --")


## Conclusion
### Choix du meilleur modèle pour la suite

En analysant [les graphiques des métriques sur MLflow](http://127.0.0.1:5000/#/experiments/9/models?viewMode=CHART), on voit bien que **XGBoost** est majoritairement le meilleur. Même s'il est moins bien que GradientBoosting sur la précision, en moyenne globale, il reste de loin le modèle le plus solide.  

On part donc sur XGBoost comme champion pour notre prochain notebook de tuning !

---
### Prochaine étape :
Génération d'artefacts visuels et recherches d'hyperparamètres sur ce XGBoost.  
[-- Ouvrir le Carnet 2 : Tuning Manuel et Artefacts --](02_Tuning_Manuel_Artefacts.ipynb)
